In [1]:
import pandas as pd

In [2]:
eg_df = pd.read_excel("2024 EGAC.xlsx", sheet_name=0)
ac_df = pd.read_excel("2024 EGAC.xlsx", skiprows=1, sheet_name=1)

In [3]:
ac_df.nunique()

Assessment Center               2191
Class Name                        11
Region                            19
Province                          96
City Name                        674
Address                         2222
Center Manager                  2180
Phone                           2218
e mail                          2180
Qualification                    197
Numberof Accredited Ass Qual       1
Assessed                         361
Certified                        369
dtype: int64

# Structural Findings
1. An institution may operate development or assessment centers across multiple cities, which means coordination must occur across different locations and personnel. For example, *TECHNO SPEAR TRAINING CENTER OF THE PHILIPPINES INCORPORATED* has centers in both Marikina City and Abucay, Bataan, so engaging with this institution would involve dealing with different staff and visiting different sites. To reflect this in a normalized structure, an institution can have multiple centers, and a city can host multiple centers, but each center is linked to exactly one institution and one city.

2. For a given center manager, attributes such as `Address`, `focal`, `phone`, and `e mail` are generally unique. A very small fraction (usually under 0.5%) show multiple values, but these do not represent different entities. Instead, they are minor formatting differences (e.g., spelling, punctuation), which were resolved by selecting the most frequent variant.

From the first and second finding, we can create this ERM Diagram for the EGAC dataset.

<div style="display: flex; justify-content: center;">
    <img src="EGAC ERM Diagram.png" alt="Centered Image" width="600">
</div>

**Figure 1.** ERM Diagram of EGAC

In [4]:
print("PROOF FOR 1ST FINDING")

# Combine eg_df and ac_df into egac_df
ac_df_cleaned = ac_df.copy()
ac_df_cleaned['Provider Name'] = ac_df_cleaned['Assessment Center']
egac_df = pd.concat([eg_df, ac_df_cleaned])
egac_df['Provider Name'] = egac_df['Provider Name'].str.strip()

# get num of institutions that operate in many cities
num_weird = (
    egac_df.groupby('Provider Name')['City Name']
    .nunique()
    .loc[lambda x : x > 1]
    .shape[0])
relative_num_weird = (num_weird / egac_df['Provider Name'].nunique()) * 100
print(f"There are {num_weird} ({relative_num_weird:.2f}%) institutions that operate in many cities.")

PROOF FOR 1ST FINDING
There are 147 (3.11%) institutions that operate in many cities.


In [103]:
egac_columns = ['Enrolled', 'Graduates', 'Assessed', 'Certified']
relative_count = summary[egac_columns].sum() / egac_df[egac_columns].sum()
absolute_count = summary[egac_columns].sum()
egac_multiple_cities = pd.DataFrame({'Absolute Count':absolute_count.to_list(), 
                                     'Relative Count':relative_count.to_list()},
                                     index=egac_columns)
egac_multiple_cities

,Absolute Count,Relative Count
Enrolled,63713.0,0.087291
Graduates,61534.0,0.091285
Assessed,94958.0,0.074956
Certified,91538.0,0.075221


In [104]:
multi_city_providers = (
    egac_df.groupby('Provider Name')['City Name']
    .nunique()
    .loc[lambda x: x > 1]
)

multi_city_list = multi_city_providers.index

multi_city_df = egac_df[
    egac_df['Provider Name'].isin(multi_city_list)
]

summary = (
    multi_city_df
    .groupby('Provider Name')
    .agg(
        Cities=('City Name', 'nunique'),
        Enrolled=('Enrolled', 'sum'),
        Graduates=('Graduates', 'sum'),
        Assessed=('Assessed', 'sum'),
        Certified=('Certified', 'sum')
    )
    .sort_values('Graduates', ascending=False)
)

summary

,Cities,Enrolled,Graduates,Assessed,Certified
Provider Name,,,,,
PHILIPPINE CALL CENTER INSTITUTE INC.,5,6201.0,7594.0,0.0,0.0
PHILIPPINE ACADEMY OF TECHNICAL STUDIES INC.,3,3235.0,2343.0,2829.0,2364.0
3A PRIME HOSPITALITY TRAINING AND ASSESSMENT CENTER INC.,2,2080.0,2009.0,0.0,0.0
"MAXIMA TECHNICAL AND SKILLS TRAINING INSTITUTE, INC.",2,2140.0,1980.0,3359.0,3121.0
"ASIAN TECH-HUB ACADEMY, INC.",4,1743.0,1963.0,0.0,0.0
...,...,...,...,...,...
"KLN SKILLS AND MANAGEMENT TRAINING AND ASSESSMENT CENTER, INC.",3,0.0,0.0,1465.0,1436.0
"GLOBAL ALLIANCE TECHNOLOGICAL INSTITUTE, CORP.",2,0.0,0.0,391.0,361.0
GENESIS INNOVATIONS AND CREATIVE DESIGN TRAINING CENTER INC.,2,0.0,0.0,265.0,227.0


In [99]:
print("PROOF FOR 2nd FINDING")
print("For DEVELOPMENT CENTERS")
dependent_cols = ['Address', 'focal', 'phone', 'e mail']
for col in dependent_cols:
    num_weird = (
        eg_df.groupby(['Provider Name', 'Province Name'])[col]
        .nunique()
        .loc[lambda x : x > 1]
        .shape[0])
    relative_num_weird = (num_weird / eg_df.shape[0]) * 100
    print(f"For the field '{col:<7}', there are {num_weird} ({relative_num_weird:.2f}%)" 
          " Provider–Province pairs with multiple values."
          )

print("-"*20)
print("FOR ASSESSMENT CENTERS")

dependent_cols = ['Address', 'Center Manager', 'Phone', 'e mail']
for col in dependent_cols:
    num_weird = (
        ac_df.groupby(['Assessment Center', 'Province'])[col]
        .nunique()
        .loc[lambda x : x > 1]
        .shape[0])
    relative_num_weird = (num_weird / ac_df.shape[0]) * 100
    print(f"For the field '{col:<7}', there are {num_weird} ({relative_num_weird:.2f}%)" 
          " Provider–Province pairs with multiple values."
          )
    
print("\nThese differences are typically minor"
      " (e.g., commas, spelling variations, or formatting inconsistencies)."
      )

PROOF FOR 2nd FINDING
For DEVELOPMENT CENTERS
For the field 'Address', there are 47 (0.51%) Provider–Province pairs with multiple values.
For the field 'focal  ', there are 25 (0.27%) Provider–Province pairs with multiple values.
For the field 'phone  ', there are 40 (0.44%) Provider–Province pairs with multiple values.
For the field 'e mail ', there are 23 (0.25%) Provider–Province pairs with multiple values.
--------------------
FOR ASSESSMENT CENTERS
For the field 'Address', there are 35 (0.10%) Provider–Province pairs with multiple values.
For the field 'Center Manager', there are 15 (0.04%) Provider–Province pairs with multiple values.
For the field 'Phone  ', there are 26 (0.08%) Provider–Province pairs with multiple values.
For the field 'e mail ', there are 28 (0.08%) Provider–Province pairs with multiple values.

These differences are typically minor (e.g., commas, spelling variations, or formatting inconsistencies).


# Quality Findings
1. In over 18% of programs (rows), the number of graduates exceeds the number of enrolled students. In some cases, programs even report graduates despite zero enrollment. This is likely due to timing differences, such as late completions being recorded after enrollment data was captured. 
Similarly, in over 12% of programs, the number of certified exceeds the number of assessed students.
2. For the assessment centers where 1350 (61.64%) do not have a classification. but after if we just let the assessment center be classified as Others, we can merge the development and assessment centers so that only 20% are identified as others.

In [110]:
print("PROOF FOR 3RD FINDING")
absolute_count = ((eg_df['Enrolled'] < eg_df['Graduates']).sum())
relative_count = ((eg_df['Enrolled'] < eg_df['Graduates']).sum()) / eg_df.shape[0]
print(f"There are {absolute_count} ({relative_count*100:.2f}%) schools which had more graduates than enrolled. "
      "This may be caused by late graduates???")

absolute_count = ((ac_df['Assessed'] < ac_df['Certified']).sum())
relative_count = ((ac_df['Assessed'] < ac_df['Certified']).sum()) / eg_df.shape[0]
print(f"There are {absolute_count} ({relative_count*100:.2f}%) schools which had more certified than assessed. "
      "I do not know why this may be caused???")

PROOF FOR 3RD FINDING
There are 1717 (18.75%) schools which had more graduates than enrolled. This may be caused by late graduates???
There are 1131 (12.35%) schools which had more certified than assessed. I do not know why this may be caused???


In [111]:
print("PROOF FOR 4th FINDING")
num_missing = ac_df['Class Name'].isna().groupby(ac_df['Assessment Center']).sum()[lambda x : x > 0].shape[0]
num_centers = ac_df['Assessment Center'].nunique()
num_missing/num_centers

PROOF FOR 4th FINDING


0.6161570059333638

In [117]:
is_HEI = (egac_df['Class Name'] == 'HEIs') | (egac_df['Class Name'] == 'Higer Education Institution (HEI)')
egac_df[is_HEI].groupby('Provider Name')[['Enrolled', 'Graduates']].sum().sort_values(by='Enrolled', ascending=False)

,Enrolled,Graduates
Provider Name,,
"NORTH CENTRAL MINDANAO COLLEGE, INC.",1677.0,2094.0
"B.E.S.T. COLLEGE OF POLOMOLOK (B.E.S.T. POLOMOLOK), INC.",1455.0,1387.0
"SOUTH CENTRAL MINDANAO COLLEGE OF SCIENCE AND TECHNOLOGY, INC.",1258.0,754.0
Colegio de Santo Cristo De Burgos Corporation,1107.0,1187.0
"Data Center College of the Philippines of Bangued, Abra, Inc.",997.0,748.0
...,...,...
"COLEGIO DE SAN CLEMENTE, INC.",0.0,0.0
SOUTHERN LUZON TECHNOLOGICAL COLLEGE FOUNDATION PILAR INC.,0.0,0.0
SAINT JOHN BOSCO COLLEGE OF NORTHERN LUZON,0.0,0.0
